# Session 14 · Case study: LLM versus trained classifier on 200 reviews

**Block 3 practice.** Compare a generative model (zero-shot and few-shot, structured output) with the trained TF-IDF classifier on the same 200 reviews: quality, cost, latency and data protection.

> **Model server.** Cells marked `REQUIRES A MODEL SERVER` call a language model through an OpenAI-compatible API. Default: a local Ollama server (`ollama pull qwen2.5:3b`, then `ollama serve`). For a hosted provider set `LLM_BASE_URL`, `LLM_MODEL` and `LLM_API_KEY` as environment variables before starting Jupyter; never write a key into the notebook. **Without a server the notebook still runs:** the LLM cells are skipped and the table shows "not run".

Theory: [03-generative-models-through-an-api.md](../theory/03-generative-models-through-an-api.md).

*Author: course team, licence CC-BY-4.0.*

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
reviews = pd.read_parquet(DATA / "train_sample.parquet")
print(reviews.shape)

## 1. The trained classifier and the 200 evaluation reviews

In [ ]:
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

texts, y = reviews["title"] + ". " + reviews["text"], reviews["label"]
X_tr, X_va, y_tr, y_va = train_test_split(texts, y, test_size=0.2, stratify=y, random_state=0)
tfidf_clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True),
                          LogisticRegression(C=4, class_weight="balanced", max_iter=2000))
tfidf_clf.fit(X_tr, y_tr)

idx = y_va.sample(200, random_state=0).index          # the same 200 reviews for every method
eval_texts, eval_y = X_va[idx], y_va[idx]
t0 = time.perf_counter()
tf_pred = pd.Series(tfidf_clf.predict(eval_texts), index=idx)
tf_sec = (time.perf_counter() - t0) / len(idx)
print(eval_y.value_counts().to_dict())

In [ ]:
def bootstrap_f1(y_true, y_pred, n_boot=2000, seed=0):
    """Macro-F1 with a 95 % bootstrap interval (Session 7)."""
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    scores = [f1_score(y_true[s], y_pred[s], average="macro")
              for s in (rng.integers(0, len(y_true), len(y_true)) for _ in range(n_boot))]
    return f1_score(y_true, y_pred, average="macro"), *np.percentile(scores, [2.5, 97.5])


print("TF-IDF: macro-F1 %.2f [%.2f, %.2f]" % bootstrap_f1(eval_y, tf_pred))

## 2. Connect to the model server

In [ ]:
import os

import httpx
from openai import OpenAI

BASE_URL = os.getenv("LLM_BASE_URL", "http://localhost:11434/v1")
MODEL = os.getenv("LLM_MODEL", "qwen2.5:3b")
API_KEY = os.getenv("LLM_API_KEY", "ollama")       # Ollama ignores the key
LOCAL = BASE_URL.startswith(("http://localhost", "http://127.0.0.1"))
client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=60)

try:                                              # is a server reachable?
    r = httpx.get(BASE_URL.rstrip("/") + "/models", headers={"Authorization": f"Bearer {API_KEY}"}, timeout=3)
    LLM_AVAILABLE = r.status_code == 200
except httpx.HTTPError:
    LLM_AVAILABLE = False
print(BASE_URL, MODEL, "server reachable:", LLM_AVAILABLE)

## 3. Prompt, schema and few-shot examples (runs offline)

In [ ]:
from typing import Literal

from pydantic import BaseModel


class Sentiment(BaseModel):
    label: Literal["neg", "neu", "pos"]


SYSTEM = (
    "You classify Amazon reviews of health and personal care products by the star rating "
    "the author most likely gave. Labels: neg = 1 or 2 stars (mainly negative); "
    "neu = 3 stars (mixed, lukewarm or 'okay'); pos = 4 or 5 stars (mainly positive). "
    'Answer only with JSON of the form {"label": "neg" | "neu" | "pos"}.'
)
RESPONSE_FORMAT = {"type": "json_schema",
                   "json_schema": {"name": "Sentiment", "schema": Sentiment.model_json_schema()}}

# few-shot examples: one per class, drawn from the TRAINING split (never from the 200)
rng_examples = X_tr[(X_tr.str.len() < 200)]
FEW_SHOT = []
for label in ["neg", "neu", "pos"]:
    example = rng_examples[y_tr[rng_examples.index] == label].sample(1, random_state=1).iloc[0]
    FEW_SHOT += [{"role": "user", "content": example},
                 {"role": "assistant", "content": Sentiment(label=label).model_dump_json()}]
for m in FEW_SHOT:
    print(m["role"], "|", m["content"][:100])

## 4. Classify the 200 reviews — REQUIRES A MODEL SERVER

With a local 3B model this takes roughly 1–5 minutes per run on a laptop; results are saved to `llm_results/`, so a rerun loads them instead of calling the model again.

In [ ]:
def classify(review, few_shot=False):
    messages = ([{"role": "system", "content": SYSTEM}] + (FEW_SHOT if few_shot else [])
                + [{"role": "user", "content": review[:4000]}])
    t0 = time.perf_counter()
    resp = client.chat.completions.create(model=MODEL, messages=messages, temperature=0,
                                          response_format=RESPONSE_FORMAT)
    sec = time.perf_counter() - t0
    try:
        label = Sentiment.model_validate_json(resp.choices[0].message.content).label
    except Exception:                       # count invalid answers instead of crashing
        label = "invalid"
    return {"label": label, "sec": sec, "tok_in": resp.usage.prompt_tokens,
            "tok_out": resp.usage.completion_tokens}


results_dir = Path("llm_results")
llm = {}
for mode in ["zero-shot", "few-shot"]:
    path = results_dir / f"{MODEL.replace(':', '_').replace('/', '_')}_{mode}.csv"
    if path.exists():
        llm[mode] = pd.read_csv(path, index_col=0)
    elif LLM_AVAILABLE:
        rows = [classify(t, few_shot=(mode == "few-shot")) for t in eval_texts]
        llm[mode] = pd.DataFrame(rows, index=idx)
        results_dir.mkdir(exist_ok=True)
        llm[mode].to_csv(path)
    else:
        print(f"{mode}: no model server, skipped")
{mode: df["label"].value_counts().to_dict() for mode, df in llm.items()}

## 5. Comparison table

In [ ]:
PRICE_IN, PRICE_OUT = 0.15, 0.60    # EXAMPLE hosted prices in USD per 1M tokens; check the provider
rows = []
f1, lo, hi = bootstrap_f1(eval_y, tf_pred)
rows.append({"method": "TF-IDF + logistic regression", "macro_F1": f1, "CI_low": lo, "CI_high": hi,
             "F1_neu": f1_score(eval_y, tf_pred, labels=["neu"], average="macro"),
             "sec_per_review": tf_sec, "USD_per_1000": 0.0, "invalid": 0,
             "data_leave_machine": "no"})
for mode in ["zero-shot", "few-shot"]:
    if mode not in llm:
        rows.append({"method": f"LLM {MODEL} {mode}", "macro_F1": "not run"})
        continue
    df = llm[mode]
    pred = df["label"].where(df["label"] != "invalid", "pos")   # count invalid answers as errors
    f1, lo, hi = bootstrap_f1(eval_y, pred)
    usd = 0.0 if LOCAL else (df["tok_in"].mean() * PRICE_IN + df["tok_out"].mean() * PRICE_OUT) / 1e6 * 1000
    rows.append({"method": f"LLM {MODEL} {mode}", "macro_F1": f1, "CI_low": lo, "CI_high": hi,
                 "F1_neu": f1_score(eval_y, pred, labels=["neu"], average="macro"),
                 "sec_per_review": df["sec"].median(), "USD_per_1000": usd,
                 "invalid": int((df["label"] == "invalid").sum()),
                 "data_leave_machine": "no" if LOCAL else "yes, to " + BASE_URL})
table = pd.DataFrame(rows).set_index("method")
table

For a local model, `USD_per_1000` is 0 but the hardware time is not: compare `sec_per_review` × 51,436 test reviews with the TF-IDF model.

## 6. Where do the methods disagree? — REQUIRES RESULTS FROM SECTION 4

In [ ]:
if "few-shot" in llm:
    cmp = pd.DataFrame({"text": eval_texts.str[:120], "true": eval_y, "tfidf": tf_pred,
                        "llm": llm["few-shot"]["label"]})
    print(classification_report(eval_y, cmp["llm"].where(cmp["llm"] != "invalid", "pos"), digits=2))
    display(cmp[cmp["tfidf"] != cmp["llm"]].head(10))
else:
    print("No LLM results: run section 4 with a model server first.")

## Your turn

1. Fill in the table with a local model (and, if your team has access, a hosted model). Are the confidence intervals of the methods overlapping?
2. Read ten reviews on which TF-IDF and the LLM disagree. Who is right more often, and in which error categories of Session 13?
3. Estimate the cost and time of classifying all 51,436 test reviews with each method.
4. Write a recommendation of five sentences for a product manager: which method would you deploy for monthly sentiment reports, and why? Address quality, cost, latency and data protection (reviews of health products can contain health data).

In [ ]:
# your notes or code here